In [3]:
import pandas as pd
import os

In [4]:
import pandas as pd
import os

# Folder path (change if needed)
folder_path = r"C:\Users\vinot\OneDrive\Desktop\All Muscle Training Exercises Dataset\All seperate"

# Empty list
all_files = []

# Loop through all csv files
for file in os.listdir(folder_path):
    if file.endswith(".csv"):
        file_path = os.path.join(folder_path, file)
        
        df = pd.read_csv(file_path)
        
        # Add muscle name column from filename
        muscle_name = file.replace("exercises_", "").replace(".csv", "")
        df["Muscle_Group"] = muscle_name
        
        all_files.append(df)

# Merge all into one dataframe
final_df = pd.concat(all_files, ignore_index=True)

# Save merged file
final_df.to_csv("merged_exercise_dataset.csv", index=False)

print("All files merged successfully")

All files merged successfully


In [5]:
final_df

,name,target,bodyPart,equipment,Muscle_Group
0,lever seated hip abduction,abductors,upper legs,leverage machine,abductors
1,side hip abduction,abductors,upper legs,body weight,abductors
2,straight leg outer hip abductor,abductors,upper legs,body weight,abductors
3,side bridge hip abduction,abductors,upper legs,body weight,abductors
4,resistance band seated hip abduction,abductors,upper legs,resistance band,abductors
...,...,...,...,...,...
163,cable floor seated wide-grip row,upper back,back,cable,upper_back
164,cable high row (kneeling),upper back,back,cable,upper_back
165,cable low seated row,upper back,back,cable,upper_back
166,cable one arm bent over row,upper back,back,cable,upper_back


# Data Cleaning

In [6]:
final_df.isna().sum()

name            0
target          0
bodyPart        0
equipment       0
Muscle_Group    0
dtype: int64

In [15]:
import pandas as pd

df = final_df.copy()

# Standardize column names
df.columns = df.columns.str.lower().str.strip()

# Remove duplicates
df = df.drop_duplicates()

# Remove null values
df = df.dropna()

# Convert text columns to lowercase
text_cols = ["name", "target", "bodypart", "equipment", "muscle_group"]
for col in text_cols:
    df[col] = df[col].str.lower()

print("Cleaned Shape:", df.shape)
df.head()

Cleaned Shape: (168, 5)


,name,target,bodypart,equipment,muscle_group
0,lever seated hip abduction,abductors,upper legs,leverage machine,abductors
1,side hip abduction,abductors,upper legs,body weight,abductors
2,straight leg outer hip abductor,abductors,upper legs,body weight,abductors
3,side bridge hip abduction,abductors,upper legs,body weight,abductors
4,resistance band seated hip abduction,abductors,upper legs,resistance band,abductors


In [17]:
df["target"].unique()

array(['abductors', 'abs', 'adductors', 'biceps', 'calves',
       'cardiovascular system', 'delts', 'forearms', 'glutes',
       'hamstrings', 'lats', 'levator scapulae', 'pectorals', 'quads',
       'serratus anterior', 'spine', 'traps', 'triceps', 'upper back'],
      dtype=object)

In [18]:
df["equipment"].unique()

array(['leverage machine', 'body weight', 'resistance band', 'assisted',
       'medicine ball', 'barbell', 'cable', 'dumbbell', 'stationary bike',
       'elliptical machine', 'stepmill machine', 'kettlebell',
       'smith machine', 'weighted', 'band', 'stability ball',
       'assisted (towel)'], dtype=object)

In [19]:
df["bodypart"].unique()

array(['upper legs', 'waist', 'upper arms', 'lower legs', 'cardio',
       'shoulders', 'lower arms', 'back', 'neck', 'chest'], dtype=object)

In [20]:
df["muscle_group"].unique()

array(['abductors', 'abs', 'adductors', 'biceps', 'calves',
       'cardiovascular_system', 'delts', 'forearms', 'glutes',
       'hamstrings', 'lats', 'levator_scapulae', 'pectorals', 'quads',
       'serratus_anterior', 'spine', 'traps', 'triceps', 'upper_back'],
      dtype=object)

In [21]:
df["name"].nunique()

168

# Fearure engineering

In [23]:
# Exercise Type
# Equipment Type Category
# Muscle Category Encoding

from sklearn.preprocessing import LabelEncoder

le_target = LabelEncoder()
le_equipment = LabelEncoder()
le_body = LabelEncoder()

df["target_encoded"] = le_target.fit_transform(df["target"])
df["equipment_encoded"] = le_equipment.fit_transform(df["equipment"])
df["bodypart_encoded"] = le_body.fit_transform(df["bodypart"])

df.head()

,name,target,bodypart,equipment,muscle_group,target_encoded,equipment_encoded,bodypart_encoded
0,lever seated hip abduction,abductors,upper legs,leverage machine,abductors,0,9,8
1,side hip abduction,abductors,upper legs,body weight,abductors,0,4,8
2,straight leg outer hip abductor,abductors,upper legs,body weight,abductors,0,4,8
3,side bridge hip abduction,abductors,upper legs,body weight,abductors,0,4,8
4,resistance band seated hip abduction,abductors,upper legs,resistance band,abductors,0,11,8


# Build Recommendation System

In [25]:
# build Content-Based Recommendation System

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Combine important columns
df["combined_features"] = df["target"] + " " + df["equipment"] + " " + df["bodypart"]

vectorizer = TfidfVectorizer()
feature_matrix = vectorizer.fit_transform(df["combined_features"])

similarity = cosine_similarity(feature_matrix)

In [26]:
# Recommendation Function

def recommend_exercises(exercise_name):
    exercise_name = exercise_name.lower()
    
    if exercise_name not in df["name"].values:
        return "Exercise not found"
    
    index = df[df["name"] == exercise_name].index[0]
    scores = list(enumerate(similarity[index]))
    
    sorted_scores = sorted(scores, key=lambda x: x[1], reverse=True)[1:6]
    
    recommended = [df.iloc[i[0]]["name"] for i in sorted_scores]
    
    return recommended

In [28]:
recommend_exercises("lever seated hip abduction")

['lever kneeling leg curl',
 'lever lying leg curl',
 'lever seated leg curl',
 'lever seated hip adduction',
 'side hip abduction']

# Clustering (Unsupervised Learning)

In [29]:
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=5, random_state=42)
df["cluster"] = kmeans.fit_predict(feature_matrix)

df[["name", "cluster"]].head()

,name,cluster
0,lever seated hip abduction,3
1,side hip abduction,2
2,straight leg outer hip abductor,2
3,side bridge hip abduction,2
4,resistance band seated hip abduction,3


### I performed exercise clustering using KMeans on TF-IDF feature representation.